# wav2vec 2.0 — Self-Supervised Learning of Speech Representations (2020)

_Papers · Speech & Audio_

**Learn speech from raw audio with no transcripts: mask latent frames and learn by a contrastive game over quantized targets; then fine-tune on tiny labeled data.**

---

This notebook is a practice scaffold for the **wav2vec 2.0 — Self-Supervised Learning of Speech Representations (2020)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
torch.manual_seed(0); np.random.seed(0)

# ---- worked example: InfoNCE contrastive term L_m (Eqn. 3) and diversity loss L_d (Eqn. 4) ----
c  = torch.tensor([1.0, 1.0]); kappa = 0.1
cands = torch.tensor([[1.0,0.9],[-1.0,0.5],[0.2,-1.0],[0.3,1.0]])   # true q_t first, then 3 distractors
sims  = F.cosine_similarity(c[None], cands)                          # cosine sim to each candidate
logits = sims / kappa
Lm = F.cross_entropy(logits[None], torch.tensor([0]))               # true target is class 0
print("sims      :", [round(s,4) for s in sims.tolist()])
print("softmax   :", [round(p,4) for p in logits.softmax(0).tolist()])
print("L_m       :", round(Lm.item(), 4))                           # -> 0.2676

pbar = torch.tensor([[0.7,0.2,0.1],[0.34,0.33,0.33]])               # G=2 groups, V=3 (avg codebook usage)
G, V = pbar.shape
Ld = (1.0/(G*V)) * (pbar * pbar.log()).sum()                        # = (1/GV) sum_g -H(p_g)
print("L_d       :", round(Ld.item(), 4))                           # -> -0.3167
print("L total   :", round((Lm + 0.1*Ld).item(), 4))               # -> 0.2359

# ---- the model: toy latent speech + quantizer + masked contrastive pretraining ----
T_LEN, D, G, Vc = 40, 16, 2, 64
codebook = torch.randn(G, Vc, D//G)                                 # fixed product-quantization codebook

def make_batch(bs, seed=None):                                      # smooth autoregressive "latent speech"
    g = torch.Generator().manual_seed(seed) if seed is not None else None
    z = torch.zeros(bs, T_LEN, D); z[:,0] = torch.randn(bs, D, generator=g)
    for t in range(1, T_LEN): z[:,t] = 0.85*z[:,t-1] + 0.5*torch.randn(bs, D, generator=g)
    return z

def quantize(z):                                                   # snap each frame to nearest code, per group
    bs,T,_ = z.shape; zg = z.view(bs,T,G,D//G); out = torch.zeros_like(zg)
    for g in range(G):
        d = (zg[:,:,g,None,:] - codebook[g][None,None]).pow(2).sum(-1)
        out[:,:,g,:] = codebook[g][d.argmin(-1)]
    return out.view(bs,T,D)

class ContextNet(nn.Module):                                       # Transformer context network
    def __init__(s):
        super().__init__(); s.proj = nn.Linear(D,64)
        enc = nn.TransformerEncoderLayer(64,4,128, batch_first=True, dropout=0.0)
        s.tr = nn.TransformerEncoder(enc,2); s.out = nn.Linear(64,D)
        s.mask_emb = nn.Parameter(torch.randn(D))                  # learned mask vector
    def forward(s, z, mask):
        zin = z.clone(); zin[mask] = s.mask_emb                    # hide masked frames
        return s.out(s.tr(s.proj(zin)))

def spans(bs, seed):                                               # mask M=3-frame spans
    m = torch.zeros(bs, T_LEN, dtype=torch.bool)
    for b in range(bs):
        for st in np.random.RandomState(seed+b).choice(T_LEN-3,5,replace=False): m[b,st:st+3]=True
    return m

K = 10                                                            # distractors -> chance = 1/(K+1) = 0.091
net = ContextNet(); opt = torch.optim.Adam(net.parameters(), lr=1.5e-3)
for ep in range(150):
    z = make_batch(48, seed=ep); q = quantize(z); m = spans(48, ep*101)
    c = net(z, m); bi,ti = torch.where(m)
    ct, qt = c[bi,ti], q[bi,ti]; M = ct.shape[0]
    di = torch.randint(0, M, (M, K))                              # distractors from other masked steps
    cand = torch.cat([qt[:,None,:], qt[di]], dim=1)               # {true + K distractors}
    sims = F.cosine_similarity(ct[:,None,:], cand, dim=-1) / 0.1  # Eqn. 3 logits
    loss = F.cross_entropy(sims, torch.zeros(M, dtype=torch.long))
    opt.zero_grad(); loss.backward(); opt.step()

# pick-the-true accuracy on a fresh batch (true target is top-1 among K distractors)
with torch.no_grad():
    zv = make_batch(48, seed=9000); qv = quantize(zv); mv = spans(48, 7777)
    cv = net(zv, mv); bi,ti = torch.where(mv); ct,qt = cv[bi,ti], qv[bi,ti]; M = ct.shape[0]
    di = torch.randint(0, M, (M, K)); cand = torch.cat([qt[:,None,:], qt[di]], 1)
    acc = (F.cosine_similarity(ct[:,None,:], cand, dim=-1).argmax(1) == 0).float().mean()
print("pick-the-true accuracy:", round(acc.item(),3), " (chance =", round(1/(K+1),3), ")")

## Visualize the data & results

_Does the masked CONTRASTIVE task teach a tiny context network to pick the true quantized target above chance — and does removing the masking kill the signal?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
torch.manual_seed(0); np.random.seed(0)
T_LEN, D, G, Vc = 40, 16, 2, 64
codebook = torch.randn(G, Vc, D//G)
def make_batch(bs, seed=None):
    g = torch.Generator().manual_seed(seed) if seed is not None else None
    z = torch.zeros(bs, T_LEN, D); z[:,0] = torch.randn(bs, D, generator=g)
    for t in range(1,T_LEN): z[:,t] = 0.85*z[:,t-1] + 0.5*torch.randn(bs, D, generator=g)
    return z
def quantize(z):
    bs,T,_ = z.shape; zg = z.view(bs,T,G,D//G); out = torch.zeros_like(zg)
    for g in range(G):
        d = (zg[:,:,g,None,:]-codebook[g][None,None]).pow(2).sum(-1)
        out[:,:,g,:] = codebook[g][d.argmin(-1)]
    return out.view(bs,T,D)
class ContextNet(nn.Module):
    def __init__(s):
        super().__init__(); s.proj=nn.Linear(D,64)
        enc=nn.TransformerEncoderLayer(64,4,128,batch_first=True,dropout=0.0)
        s.tr=nn.TransformerEncoder(enc,2); s.out=nn.Linear(64,D)
        s.mask_emb=nn.Parameter(torch.randn(D))
    def forward(s,z,mask):
        zin=z.clone(); zin[mask]=s.mask_emb; return s.out(s.tr(s.proj(zin)))
def spans(bs,seed):
    m=torch.zeros(bs,T_LEN,dtype=torch.bool)
    for b in range(bs):
        for st in np.random.RandomState(seed+b).choice(T_LEN-3,5,replace=False): m[b,st:st+3]=True
    return m
K=10
def run(use_mask, epochs=150):
    torch.manual_seed(0)
    net=ContextNet(); opt=torch.optim.Adam(net.parameters(),lr=1.5e-3); accs=[]
    for ep in range(epochs):
        z=make_batch(48,seed=ep); q=quantize(z)
        m = spans(48,ep*101) if use_mask else torch.zeros(48,T_LEN,dtype=torch.bool)
        c=net(z,m)
        if use_mask:
            bi,ti=torch.where(m); ct,qt=c[bi,ti],q[bi,ti]; M=ct.shape[0]
            di=torch.randint(0,M,(M,K)); cand=torch.cat([qt[:,None,:],qt[di]],1)
            loss=F.cross_entropy(F.cosine_similarity(ct[:,None,:],cand,dim=-1)/0.1,
                                 torch.zeros(M,dtype=torch.long))
        else:
            loss=(c-q).pow(2).mean()                      # ABLATION: reconstruct, no masking
        opt.zero_grad(); loss.backward(); opt.step()
        if ep%15==0 or ep==epochs-1:
            with torch.no_grad():
                zv=make_batch(48,seed=9000); qv=quantize(zv); mv=spans(48,7777)
                cv=net(zv,mv); bi,ti=torch.where(mv); ct,qt=cv[bi,ti],qv[bi,ti]; M=ct.shape[0]
                di=torch.randint(0,M,(M,K)); cand=torch.cat([qt[:,None,:],qt[di]],1)
                accs.append((ep, round((F.cosine_similarity(ct[:,None,:],cand,dim=-1).argmax(1)==0)
                                       .float().mean().item(),4)))
    return accs
print("chance =", round(1/(K+1),4))
print("masked+contrastive:", run(True))
print("ablation (no mask):", run(False))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compute $\mathcal{L}_m$ for a candidate set of 3 (true + 2 distractors) with logits (after sim/$\kappa$) $= (4,\,1,\,0)$, true target first.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Exponentiate: $e^4=54.60,\ e^1=2.718,\ e^0=1$. — _Softmax numerator/denominator pieces._
- Sum $=58.32$; true-target softmax $=54.60/58.32=0.9362$. — _Probability mass on the true target._
- $\mathcal{L}_m=-\log(0.9362)$. — _InfoNCE is $-\log$ of the true target's softmax probability._

**Answer:** $\mathcal{L}_m=-\log(0.9362)\approx 0.0659$. A confident correct pick &rarr; small loss.

</details>

**Problem 2.** A codebook group's average usage is $\bar{p}=(0.5,\,0.5)$ (V=2). What is its entropy, and is this good or bad for $\mathcal{L}_d$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $H=-(0.5\log 0.5+0.5\log 0.5)=\log 2=0.693$. — _Uniform over 2 entries gives maximal entropy._
- Compare to a spike $(1,0)$: $H=0$. — _A collapsed codebook has zero entropy._

**Answer:** $H=0.693$ (maximal for V=2). Maximal entropy &rarr; most negative $-H$ &rarr; LOWEST $\mathcal{L}_d$ &rarr; this is the GOOD case (even code use).

</details>

**Problem 3.** Ablation. If you remove masking entirely (no frame is hidden) before the contrastive task, what happens, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- With nothing masked, the context network can just copy each input latent to its output. — _No information is removed, so $\mathbf{c}_t$ needs no inference._
- There is no held-out frame to predict &rarr; the contrastive game has no real difficulty. — _The task degenerates; the model learns no predictive structure._

**Answer:** Pick-the-true-target accuracy stays at chance $\frac{1}{K+1}$ &mdash; matching our CODEVIZ ablation curve. Masking is what creates the learning signal.

</details>